In [ ]:
import pandas as pd
df = pd.read_csv("..\\data\\student-course-completion-prediction-dataset-trimmed.csv")

print(df.shape)
print(df.head())

print(df.dtypes.to_frame(name="dtype"))

In [ ]:
# Keep an untouched copy of the loaded and named dataset
df_copy = df.copy(deep=True)

In [ ]:
# Check missing values
missing = df_copy.isna().sum()
print("Columns containing missing values:")
print(missing[missing > 0].sort_values(ascending=False))

# Check duplicate rows
print("\nNumber of duplicate rows:")
print(df_copy.duplicated().sum())

# Check the ranges of important measurements
range_cols = [
    "Age", 
    "Course_Duration_Days", 
    "Average_Session_Duration_Min", 
    "Video_Completion_Rate", 
    "Time_Spent_Hours",
    "Days_Since_Last_Login",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Quiz_Score_Avg", 
    "Project_Grade", 
    "Progress_Percentage" 
]

print("\nMinimum and maximum values:")
print(df_copy[range_cols].agg(["min", "max"]))

In [ ]:
review_cols = [
    "Age", # improbable to have students younger than 10 or older than 90
    "Course_Duration_Days", # 0 <= result <= 90
    "Average_Session_Duration_Min", # 0 <= result <= 80
    "Video_Completion_Rate", # 0 <= result <= 100
    "Time_Spent_Hours", # 0 <= result <= 30
    "Days_Since_Last_Login", # 0 <= result <= 100
    "Assignments_Submitted", # submitted + missed assignments total should not exceed 10
    "Assignments_Missed", # submitted + missed assignments total should not exceed 10 
    "Quiz_Attempts", # 0 <= result <= 16
    "Quiz_Score_Avg", # 0 <= result <= 100
    "Project_Grade", # 0 <= result <= 100
    "Progress_Percentage" # 0 <= result <= 100
]

records_to_review = df_copy.loc[
    ((df_copy["Age"] < 10) | (df_copy["Age"] > 90)) |
    ((df_copy["Course_Duration_Days"] < 0) | (df_copy["Course_Duration_Days"] > 90)) |
    ((df_copy["Average_Session_Duration_Min"] < 0) | (df_copy["Average_Session_Duration_Min"] > 80)) |
    ((df_copy["Video_Completion_Rate"] < 0) | (df_copy["Video_Completion_Rate"] > 100)) |
    ((df_copy["Time_Spent_Hours"] < 0) | (df_copy["Time_Spent_Hours"] > 30)) |
    ((df_copy["Days_Since_Last_Login"] < 0) | (df_copy["Days_Since_Last_Login"] > 100)) |
    ((df_copy["Assignments_Submitted"] < 0) | (df_copy["Assignments_Submitted"] > 10)) |
    ((df_copy["Assignments_Missed"] < 0) | (df_copy["Assignments_Missed"] > 10)) |
    ((df_copy["Quiz_Attempts"] < 0) | (df_copy["Quiz_Attempts"] > 16)) |
    ((df_copy["Quiz_Score_Avg"] < 0) | (df_copy ["Quiz_Score_Avg"] > 100)) |
    ((df_copy["Project_Grade"] < 0) | (df_copy["Project_Grade"] > 100)) |
    ((df_copy["Progress_Percentage"] < 0) | (df_copy["Progress_Percentage"] > 100))
].sort_values(
    ["Age", "Course_Duration_Days", "Average_Session_Duration_Min"]
)

print(records_to_review.to_string())

In [ ]:
def clean_data(data, invaled_record):
    # Create a copy so the original data stays unchanged
    clean = data.copy(deep=True)

    # Correct the confirmed age error for STU100001
    # The original dataset shows age 17, not 170
    clean.loc[
        clean["Student_ID"] == "STU100001",
        "Age"
    ] = 17
    

    # Display the rows that will be removed
    print("Unreliable records removed:")
    print(
        clean.loc[
            invalid_record,
            [
                "Student_ID",
                "Name",
                "Gender",
                "Age",
                "Education_Level",
                "Employment_Status",
                "Completed"
            ]
        ]
    )

    # Keep only the valid rows
    clean = clean.loc[~invalid_record].copy()

    for col in [
        "Quiz_Score_Avg",
        "Project_Grade",
        "Progress_Percentage",
        "App_Usage_Percentage"
    ]:
        clean[col] = clean[col].fillna(clean[col].median())

    clean = clean.drop_duplicates()
    clean = clean.reset_index(drop=True)

    return clean

In [ ]:
# Apply the cleaning function to the copy of the original data
df_clean_preview = clean_data(df_copy, records_to_review)

In [ ]:
print("Original shape:", df_copy.shape)
print("Cleaned shape:", df_clean_preview.shape)

print(
    "Rows removed:",
    len(df_copy) - len(df_clean_preview)
)

In [ ]:
range_cols = [
    "Age", 
    "Course_Duration_Days", 
    "Average_Session_Duration_Min", 
    "Video_Completion_Rate", 
    "Time_Spent_Hours",
    "Days_Since_Last_Login",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Quiz_Score_Avg", 
    "Project_Grade", 
    "Progress_Percentage" 
]

print("\nMinimum and maximum values:")
print(df_copy[range_cols].agg(["min", "max"]))

In [ ]:
invalid_record = (
        ((df_clean_preview["Age"] < 10) | (df_clean_preview["Age"] > 90)) |
        ((df_clean_preview["Course_Duration_Days"] < 0) | (df_clean_preview["Course_Duration_Days"] > 90)) |
        ((df_clean_preview["Average_Session_Duration_Min"] < 0) | (df_clean_preview["Average_Session_Duration_Min"] > 80)) |
        ((df_clean_preview["Video_Completion_Rate"] < 0) | (df_clean_preview["Video_Completion_Rate"] > 100)) |
        ((df_clean_preview["Time_Spent_Hours"] < 0) | (df_clean_preview["Time_Spent_Hours"] > 30)) |
        ((df_clean_preview["Days_Since_Last_Login"] < 0) | (df_clean_preview["Days_Since_Last_Login"] > 100)) |
        ((df_clean_preview["Assignments_Submitted"] < 0) | (df_clean_preview["Assignments_Submitted"] > 10)) |
        ((df_clean_preview["Assignments_Missed"] < 0) | (df_clean_preview["Assignments_Missed"] > 10)) |
        ((df_clean_preview["Quiz_Attempts"] < 0) | (df_clean_preview["Quiz_Attempts"] > 16)) |
        ((df_clean_preview["Quiz_Score_Avg"] < 0) | (df_clean_preview["Quiz_Score_Avg"] > 100)) |
        ((df_clean_preview["Project_Grade"] < 0) | (df_clean_preview["Project_Grade"] > 100)) |
        ((df_clean_preview["Progress_Percentage"] < 0) | (df_clean_preview["Progress_Percentage"] > 100))
    )

print(df_clean_preview.loc[invalid_record].to_string())

In [ ]:
missing = df_clean_preview.isna().sum()
print("\nColumns still containing missing values:")
print(
    missing[missing > 0]
    .sort_values(ascending=False)
)
print(
    "\nDuplicate rows:",
    df_clean_preview.duplicated().sum()
)

In [ ]:
print("\nFirst five cleaned records:")
print(
    df_clean_preview[
        [
            "Student_ID",
            "Name",
            "Gender",
            "Age",
            "Education_Level",
            "Employment_Status",
            "Completed"
        ]
    ].head()
)

In [ ]:
import sqlite3
# Use the cleaned data already reviewed above
df_etl_clean = df_clean_preview.copy(deep=True)

with sqlite3.connect("..\\data\\etl.db") as conn:
    df_etl_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ETL completed.")
print("Cleaned table shape:", df_etl_clean.shape)

In [ ]:
with sqlite3.connect("..\\data\\elt.db") as conn:
    # Load the untouched raw data first
    df_copy.to_sql(
        "raw",
        conn,
        if_exists="replace",
        index=False
    )
    # Read the raw table back into Pandas
    df_elt_raw = pd.read_sql(
        "SELECT * FROM raw",
        conn
    )
    # Transform the data after loading
    df_elt_clean = clean_data(df_elt_raw)
    # Store the cleaned result as a second table
    df_elt_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ELT completed.")
print("Raw table shape:", df_elt_raw.shape)
print("Cleaned table shape:", df_elt_clean.shape)

In [ ]:
with sqlite3.connect("..\\data\\etl.db") as conn:
    etl_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
print("Tables in etl.db:")
print(etl_tables)

with sqlite3.connect("..\\data\\elt.db") as conn:
    elt_tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
print("\nTables in elt.db:")
print(elt_tables)

In [ ]:
print(df_copy.shape)
print(df_copy.dtypes.value_counts())
print(df_copy.info())
print(df_copy.describe())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df_etl_clean["App_Usage_Percentage"].dropna(), bins=20, edgecolor="white", linewidth=1)
plt.xlabel("Application Usage Percentage"); plt.ylabel("Number of Students")
plt.title("Distribution of Application Usage Percentage")
plt.show()

In [ ]:
plt.hist(df_etl_clean["Age"].dropna(), bins=25, edgecolor="white", linewidth=1)
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.title("Distribution of Age")
plt.show()

In [ ]:
plt.hist(df_copy["Age"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Age (before cleaning)")
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_copy["App_Usage_Percentage"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Application Usage Percentage (before cleaning)")
plt.xlabel("Application Usage Percentage")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_copy["Project_Grade"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Project Grade (before cleaning)")
plt.xlabel("Project Grade")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# Compare the variables that were reviewed during cleaning
for col in ["Age", "Project_Grade", "Quiz_Score_Avg", "Time_Spent_Hours"]:
    print(
        col,
        "raw range:",
        (df_copy[col].min(), df_copy[col].max()),
        "clean range:",
        (df_etl_clean[col].min(), df_etl_clean[col].max())
    )

In [ ]:
plt.hist(df_etl_clean["Age"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Age (after cleaning)")
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_etl_clean["App_Usage_Percentage"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Application Usage Percentage (after cleaning)")
plt.xlabel("Application Usage Percentage")
plt.ylabel("Number of Students")
plt.show()
plt.hist(df_etl_clean["Project_Grade"].dropna(), bins=30, edgecolor="white", linewidth=1)
plt.title("Project Grade (after cleaning)")
plt.xlabel("Project Grade")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
plt.boxplot(df_etl_clean["Age"].dropna(), orientation="horizontal", tick_labels=["Age"])
plt.xlabel("Age")
plt.show()

In [ ]:
def iqr_flag(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

age_flag = iqr_flag(df_etl_clean["age"])
height_flag = iqr_flag(df_etl_clean["height"])
weight_flag = iqr_flag(df_etl_clean["weight"])
demographic_flags = age_flag | height_flag | weight_flag

print(
    df_etl_clean.loc[
        demographic_flags,
        ["age", "sex", "height", "weight", "class"]
    ].sort_values(["age", "height", "weight"]).to_string()
)

# Heart rate is checked separately with a z-score
mean_hr = df_etl_clean["heart_rate"].mean()
std_hr = df_etl_clean["heart_rate"].std()
z_hr = (df_etl_clean["heart_rate"] - mean_hr) / std_hr

print("Heart-rate values with |z| > 3:")
print(df_etl_clean.loc[z_hr.abs() > 3, ["age", "sex", "heart_rate", "class"]])

In [ ]:
print(
    df_etl_clean.loc[
        (df_etl_clean["age"] == 75) &
        (df_etl_clean["sex"] == 0) &
        (df_etl_clean["height"] == 190) &
        (df_etl_clean["weight"] == 80)
    ].to_string()
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].boxplot(df_etl_clean["height"].dropna(), tick_labels=["Height"])
axes[0].set_title("Height"); axes[0].set_ylabel("cm")
axes[1].boxplot(df_etl_clean["heart_rate"].dropna(), tick_labels=["Heart rate"])
axes[1].set_title("Heart rate"); axes[1].set_ylabel("bpm")
axes[2].boxplot(df_etl_clean["weight"].dropna(), tick_labels=["Weight"])
axes[2].set_title("Weight"); axes[2].set_ylabel("kg")
plt.suptitle("Cleaned data before optional capping")
plt.show()

In [ ]:
df_sensitivity = df_etl_clean.copy(deep=True)
df_sensitivity["height_capped"] = df_sensitivity["height"].clip(
    lower=df_sensitivity["height"].quantile(0.01)
)
df_sensitivity["heart_rate_capped"] = df_sensitivity["heart_rate"].clip(
    lower=df_sensitivity["heart_rate"].quantile(0.01),
    upper=df_sensitivity["heart_rate"].quantile(0.99)
)
df_sensitivity["weight_capped"] = df_sensitivity["weight"].clip(
    upper=df_sensitivity["weight"].quantile(0.99)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].boxplot(df_sensitivity["height_capped"].dropna(), tick_labels=["Height"])
axes[0].set_title("Height (sensitivity-capped)"); axes[0].set_ylabel("cm")
axes[1].boxplot(df_sensitivity["heart_rate_capped"].dropna(), tick_labels=["Heart rate"])
axes[1].set_title("Heart rate (capped)"); axes[1].set_ylabel("bpm")
axes[2].boxplot(df_sensitivity["weight_capped"].dropna(), tick_labels=["Weight"])
axes[2].set_title("Weight (capped)"); axes[2].set_ylabel("kg")
plt.suptitle("Optional sensitivity-only capped view")
plt.show()

In [ ]:
rate = df_etl_clean.groupby("sex", observed=False)["arrhythmia_present"].mean()
print(rate)
sex_labels = {0: "Male", 1: "Female"}
labels = [sex_labels[int(value)] for value in rate.index]
plt.bar(labels, rate.values)
plt.xlabel("Sex")
plt.ylabel("Proportion with arrhythmia")
plt.title("Arrhythmia rate by sex")
plt.show()          # Figure 13 is what you'll see

In [ ]:
import numpy as np

num_cols = ["age", "height", "weight", "qrs_duration", "pr_interval", "qt_interval",
    "heart_rate"]
corr_matrix = df_etl_clean[num_cols].corr()
print(corr_matrix.round(2))

# Find the strongest non-diagonal linear association
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
strongest_pair = upper_triangle.abs().stack().idxmax()
print(
    "Strongest absolute correlation:",
    strongest_pair,
    round(corr_matrix.loc[strongest_pair[0], strongest_pair[1]], 2)
)
plt.imshow(corr_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation between 7 of 279 input features")
plt.show()          # Figure 14 is what you'll see

In [ ]:
class_names = {
    1: "Normal", 2: "Ischemic changes (CAD)", 3: "Old Ant. MI", 4: "Old Inf. MI",
    5: "Sinus tachycardia", 6: "Sinus bradycardia", 7: "PVC", 8: "Supraventricular PC",
    9: "Left bundle branch block", 10: "Right bundle branch block",
    11: "1st degree AV block", 12: "2nd degree AV block", 13: "3rd degree AV block",
    14: "Left vent. hypertrophy", 15: "Atrial fib./flutter", 16: "Other",
}
pqrst_cols = ["p_interval", "qrs_duration", "pr_interval", "qt_interval", "t_interval",
    "heart_rate"]
class_counts = df_etl_clean["class"].value_counts().sort_index()
print("Patients per class:")
print(class_counts)
profile = df_etl_clean.groupby("class")[pqrst_cols].mean()
print("Mean profile per class:")
print(profile.round(1))
# standardise each column so differing scales (ms vs bpm) do not dominate the color map
profile_z = (profile - profile.mean()) / profile.std()
row_labels = [class_names[int(c)] for c in profile_z.index]
plt.imshow(profile_z.values, cmap="RdYlGn", vmin=-2, vmax=2, aspect="auto")
plt.xticks(range(len(pqrst_cols)), pqrst_cols, rotation=45, ha="right")
plt.yticks(range(len(profile_z)), row_labels)
plt.colorbar(label="Standardised mean (per column)")
plt.title("PQRST + heart rate profile, by arrhythmia type")
plt.show()

In [ ]:
miss = df_copy.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0] * 100
plt.barh(miss.index.astype(str), miss.values)
plt.xlabel("% missing")
plt.title("Missing values in the raw data")
plt.show()

In [ ]:
order = sorted(df_etl_clean["class"].unique())
x_pos = df_etl_clean["class"].map({c: i for i, c in enumerate(order)})
plt.scatter(x_pos, df_etl_clean["heart_rate"], alpha=0.4)
means = df_etl_clean.groupby("class")["heart_rate"].mean()
print("Mean heart rate by class:")
print(means.sort_values())
plt.scatter(range(len(order)), [means[c] for c in order], color="red", marker="D",
    label="Mean per type")
plt.xticks(range(len(order)), [class_names[c] for c in order], rotation=45, ha="right")
plt.xlabel("Arrhythmia type"); plt.ylabel("Heart rate (bpm)")
plt.title("Heart rate by arrhythmia type")
plt.legend()
plt.show()

In [ ]:
from matplotlib.lines import Line2D

colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
plt.scatter(df_etl_clean["height"], df_etl_clean["weight"], c=colors, alpha=0.5)
plt.xlabel("Height (cm)")
plt.ylabel("Weight (kg)")
plt.title("Height vs weight, by diagnosis")
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="teal", markersize=8, label="No arrhythmia"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="orange", markersize=8, label="Arrhythmia present"),
]
plt.legend(handles=legend_handles)
plt.show()

In [ ]:
jitter = df_etl_clean["arrhythmia_present"].astype(int) + np.random.uniform(-0.08, 0.08,
    len(df_etl_clean))
colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
plt.scatter(df_etl_clean["age"], jitter, c=colors, alpha=0.5)
plt.yticks([0, 1], ["No", "Yes"])
plt.xlabel("Age")
plt.ylabel("arrhythmia_present")
plt.title("Age vs arrhythmia_present")
plt.show()          # y-axis labels (No/Yes) already identify the two colours here

In [ ]:
colors = df_etl_clean["arrhythmia_present"].map({True: "orange", False: "teal"})
cols = ["height", "weight", "qt_interval", "heart_rate"]
fig, axes = plt.subplots(4, 4, figsize=(9, 9))
for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        ax = axes[i, j]
        if i == j:
            ax.hist(df_etl_clean[c1].dropna(), bins=20)
        else:
            ax.scatter(df_etl_clean[c2], df_etl_clean[c1], s=4, alpha=0.35, c=colors)
        if i == 3:
            ax.set_xlabel(c2)
        if j == 0:
            ax.set_ylabel(c1)
plt.tight_layout()
from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(color="teal", label="No arrhythmia"),
    Patch(color="orange", label="Arrhythmia present"),
], loc="upper right")
plt.show()

In [ ]:
from scipy import stats

a = df_etl_clean.loc[df_etl_clean["arrhythmia_present"], "heart_rate"]
b = df_etl_clean.loc[~df_etl_clean["arrhythmia_present"], "heart_rate"]
t, p = stats.ttest_ind(a, b, equal_var=False)

print("Mean with arrhythmia:", a.mean())
print("Mean without arrhythmia:", b.mean())

print(f"t = {t:.2f}, p = {p:.4f}")

if p < 0.05:
    print("Reject H0: the mean heart rates differ significantly.")
else:
    print("Fail to reject H0: no significant mean difference was detected.")

In [ ]:
from scipy import stats

table = pd.crosstab(
    df_etl_clean["sex"],
    df_etl_clean["arrhythmia_present"]
)
print(table)
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.2f}, p = {p:.6f}")
if p < 0.05:
    print("Reject H0: sex and diagnosis are associated in this sample.")
else:
    print("Fail to reject H0: no significant association was detected.")

In [ ]:
# Check duplicate Student_ID values
print("Duplicate Student_IDs:", df["Student_ID"].duplicated().sum())

In [ ]:
# Display all rows that share a duplicated Student_ID
# keep=False shows both copies of the duplicated record
df[df["Student_ID"].duplicated(keep=False)]

In [ ]:
# Create a cleaned copy and remove completely identical duplicate rows
df_clean = df.drop_duplicates().copy()

In [ ]:
# Compare the number of rows before and after duplicate removal
print("Before:", df.shape)
print("After:", df_clean.shape)

In [ ]:
# Check that there are no completely duplicated rows remaining
print(
    "Duplicate rows:",
    df_clean.duplicated().sum()
)

# Check that there are no duplicated Student_ID values remaining
print(
    "Duplicate Student_IDs:",
    df_clean["Student_ID"].duplicated().sum()
)

In [ ]:
# Show summary statistics for the Age column
# This helps us inspect the age range before deciding
# whether my teammate's current age-cleaning rule is justified
df_copy["Age"].describe()

In [ ]:
# Show all records where Age is greater than 60
# This lets us inspect the unusual ages before deciding what to clean
df_copy.loc[
    df_copy["Age"] > 60,
    ["Student_ID", "Name", "Age"]
].sort_values("Age")

In [ ]:
# Verify that the confirmed age error was corrected
# Shows STU100001 with Age = 17
df_clean_preview.loc[
    df_clean_preview["Student_ID"] == "STU100001",
    ["Student_ID", "Name", "Age"]
]

In [48]:
# Check the total number of assignments for each student
# This helps us see whether submitted + missed is consistent across the dataset
assignment_total = (
    df_copy["Assignments_Submitted"]
    + df_copy["Assignments_Missed"]
)

# Show how many students have each total
print(assignment_total.value_counts().sort_index())

9      3051
10    26950
Name: count, dtype: int64
